# Part 8 - Stopping the Runaway: budgets, loop detection, and the circuit breaker

*Agents from First Principles, Part 8.*

Every part so far made the agent more capable. This one makes it safe to leave running. The loop from Part 1 ends only when the controller calls `finish()` or it hits a single `max_steps` cap (RAG Part 19's one guard). A real controller does not always converge. It can get stuck re-issuing the SAME search forever, or it can make steady, plausible, never-ending progress that quietly runs up a large bill. An agent that does not know when to give up is a liability.

A single step cap is not enough, for two reasons. It cannot tell a run that is making progress from one spinning in place, and it ignores the dimensions that actually cost money: tokens, dollars, and time. This part adds three guards.

1. **A multi-dimensional budget.** A `BudgetMeter` tracks steps, estimated tokens, estimated USD, and (simulated) wall-clock, each with its own ceiling, checked BEFORE every step. The first dimension to cross its ceiling stops the run. One number cannot capture "too expensive."

2. **A loop detector.** Repeating the identical `(action, args)` is the signature of a stuck agent. The detector counts repeats and flags a loop at a threshold. This FORMALIZES the informal note from RAG Part 19 ("if the agent runs the same search twice, stop") into a real, reusable check.

3. **A circuit breaker.** When the budget is exceeded or a loop is detected, the breaker trips: closed -> tripped -> graceful-finish. Crucially, tripping does not crash. It returns the best PARTIAL result the agent has, plus the reason it stopped, so the caller gets something useful and an explanation.

These are not throwaway demos. The `BudgetMeter` is reused in **Part 13** (to cap a code-execution sandbox) and the circuit breaker in **Part 14** (to stop a runaway supervisor). We build them carefully here because later parts depend on them.

This part GENERALIZES RAG Part 19's single `max_steps` plus its informal loop note into multi-dimensional budgets, a real cycle detector, and a graceful breaker. It does not re-teach the reason/act/observe loop; it references it (the recovery and limits theme from Part 5).

> **Runs with no network, no API key, and no third-party dependency.** Deterministic controllers are the offline default; `generate()` is the real-LLM path one flag away. Set `OPENAI_API_KEY` to see the real banner; the code always falls through to the deterministic controllers so output stays reproducible.

Companion script: `budget_circuit_breaker.py`

## Setup, and what is real versus illustrative

A single standard-library import does the work: `os` lets us check for an API key without ever requiring one. Everything else is plain Python, so every cell runs fully offline, exactly as in Parts 1-7.

Be precise about what the numbers mean, because honesty matters more than a dramatic demo:

- **Wall-clock is SIMULATED.** Each step adds a fixed estimate (`~1.2s`), not a measured duration, so the run is reproducible. A real system would measure elapsed time.
- **Cost is ILLUSTRATIVE.** The per-token price and per-step token count are made-up estimates for the demo, not a real price sheet.
- **The runaway-cost framing is GENERALIZED.** There are documented incidents of unbounded agent loops burning a serious amount of money before anyone noticed; we do not attach a fabricated specific dollar figure to any of them. The point stands without one: an unguarded loop can be expensive.

In [ ]:
import os

print("ready -- no network, no API key, no third-party dependency required")

## Step 1 - The `BudgetMeter`: multi-dimensional, checked before every step

A single `max_steps` cap answers only one question: how many turns? It cannot express "too many tokens" or "too costly" or "too slow." The `BudgetMeter` tracks four dimensions at once, each with its own ceiling.

- `charge_step()` advances every dimension by its per-step estimate (one step, `TOKENS_PER_STEP` tokens, the matching USD, and `WALL_PER_STEP` simulated seconds).
- `exceeded()` returns the FIRST dimension over its limit (or `None`). That ordering is deliberate: the run stops on whichever ceiling it crosses first, and the caller learns which one.
- `snapshot()` renders the current meter for the trace.

The constants are illustrative, as noted above: `USD_PER_TOKEN` is a stand-in price, `TOKENS_PER_STEP` a stand-in token estimate, and `WALL_PER_STEP` a simulated, not measured, duration.

In [ ]:
USD_PER_TOKEN = 0.00001        # an illustrative per-token price, not a real sheet
TOKENS_PER_STEP = 80           # an illustrative per-step token estimate
WALL_PER_STEP = 1.2            # SIMULATED seconds per step (not measured)


class BudgetMeter:
    def __init__(self, max_steps, max_tokens, max_usd, max_wall):
        self.max_steps, self.max_tokens = max_steps, max_tokens
        self.max_usd, self.max_wall = max_usd, max_wall
        self.steps = self.tokens = 0
        self.usd = self.wall = 0.0

    def charge_step(self):
        self.steps += 1
        self.tokens += TOKENS_PER_STEP
        self.usd += TOKENS_PER_STEP * USD_PER_TOKEN
        self.wall += WALL_PER_STEP

    def exceeded(self):
        if self.steps >= self.max_steps:
            return f"step budget ({self.max_steps})"
        if self.tokens >= self.max_tokens:
            return f"token budget ({self.max_tokens})"
        if self.usd >= self.max_usd:
            return f"cost budget (${self.max_usd:.2f})"
        if self.wall >= self.max_wall:
            return f"time budget ({self.max_wall:.0f}s)"
        return None

    def snapshot(self):
        return (f"steps {self.steps} | tokens {self.tokens} | "
                f"${self.usd:.4f} | ~{self.wall:.1f}s")


print("BudgetMeter ready.")

## Step 2 - The `LoopDetector`: count identical actions, flag a loop at a threshold

RAG Part 19 had an informal note: "if the agent runs the same search twice, stop." That instinct was correct but never turned into code. Here it becomes a reusable check. `record(action_key)` increments the count for an `(action, args)` key and returns `(is_loop, count)`, where `is_loop` is true once the same key has been seen `threshold` times.

We key on EXACT repeats of the action plus its arguments. Alternating A-B-A-B cycles are a natural extension of the same idea; keying on exact repeats keeps the mechanism small and readable while catching the common stuck-agent signature.

In [ ]:
class LoopDetector:
    def __init__(self, threshold=3):
        self.threshold = threshold
        self.counts = {}

    def record(self, action_key):
        self.counts[action_key] = self.counts.get(action_key, 0) + 1
        n = self.counts[action_key]
        return (n >= self.threshold), n


print("LoopDetector ready.")

## Step 3 - The `CircuitBreaker`: closed -> tripped -> graceful-finish

The breaker is deliberately tiny: a state (`closed` or `tripped`) and a `reason`. The important property is what it does NOT do. It does not raise. When `trip(reason)` is called, it records WHY it stopped, and the agent loop reads that to return its best PARTIAL result instead of crashing or spinning. A crash loses everything the agent already gathered; a graceful trip keeps it and explains itself.

In [ ]:
class CircuitBreaker:
    def __init__(self):
        self.state = "closed"
        self.reason = None

    def trip(self, reason):
        self.state = "tripped"
        self.reason = reason


print("CircuitBreaker ready.")

## Step 4 - Two controllers that DO NOT converge

These are the two failure shapes a single step cap cannot tell apart. Neither ever calls `finish()` -- that absence is exactly the bug we are guarding against.

- `stuck_controller`: re-issues the IDENTICAL search every step. The query never changes, so re-searching cannot help; the loop detector is what catches this.
- `wandering_controller`: a DIFFERENT, plausible search every step, forever. No two actions repeat, so the loop detector sees nothing; the BUDGET is what catches this.

`search_policy` is the tool both call. A "discount code" query returns an empty-ish result ("no discount code on file"), so re-searching it is hopeless; any other query returns plausible clause text, which is what makes the wandering agent look like it is making progress.

In [ ]:
def stuck_controller(goal, steps):
    return ("Look up the discount code again.",
            "search_policy", {"query": "discount code for ORD-9999"})


def wandering_controller(goal, steps):
    n = len(steps) + 1
    return (f"Audit policy clause {n}.",
            "search_policy", {"query": f"policy clause {n}"})


def search_policy(query):
    if "discount code" in query:
        return "(no discount code on file)"          # empty-ish: re-searching cannot help
    return f"clause text for '{query}'"


print("stuck_controller, wandering_controller, search_policy ready.")

## Step 5 - The guarded agent loop

This is where the three guards come together. Before every step the loop asks the breaker whether the budget is exceeded; after choosing an action it asks the loop detector. Either one trips the breaker, which ends the run with a graceful PARTIAL result rather than a crash.

The `guards` flag lets us run the SAME loop with protection off, falling back to a plain `hard_cap` so we can watch an unguarded agent run away (and cut it off at the demo cap so the notebook still terminates). With `guards=True`:

- The **pre-step budget guard** calls `meter.exceeded()`. If a ceiling is crossed, the breaker trips and the loop stops BEFORE spending another step.
- The **loop detector** records `(tool, args)` and trips the breaker the moment a key hits the threshold.

`_graceful_partial` builds the return value. If the breaker never tripped (the unguarded path), it reports the run as a RUNAWAY. If it did trip, it returns what was gathered plus the reason, never an exception.

In [ ]:
def run_agent(goal, controller, ceilings, guards=True, loop_threshold=3,
              hard_cap=10, trace=True):
    meter = BudgetMeter(*ceilings)
    detector = LoopDetector(threshold=loop_threshold)
    breaker = CircuitBreaker()
    steps = []

    def log(msg):
        if trace:
            print(msg)

    while True:
        # --- pre-step budget guard -----------------------------------------
        if guards:
            over = meter.exceeded()
            if over:
                breaker.trip(f"{over} reached")
                log(f"    BREAKER TRIPPED before step {meter.steps + 1}: {breaker.reason}")
                break
        else:
            if meter.steps >= hard_cap:
                log(f"    [demo cap] cut off at {hard_cap} steps; an unguarded agent would "
                    "not stop on its own.")
                break

        thought, tool, args = controller(goal, steps)
        if tool == "finish":
            log(f"    finish -> {args.get('answer')}")
            return args.get("answer"), meter, breaker

        # --- loop detection on (action, args) ------------------------------
        key = (tool, tuple(sorted(args.items())))
        is_loop, count = detector.record(key)

        meter.charge_step()
        obs = search_policy(args["query"])
        steps.append((tool, args, obs))
        rep = f"  (x{count})" if count > 1 else ""
        log(f"    step {meter.steps}: {tool}({args['query']!r}) -> {obs}{rep}    [{meter.snapshot()}]")

        if guards and is_loop:
            breaker.trip(f"loop detected: identical action repeated {count} times")
            log(f"    BREAKER TRIPPED: {breaker.reason}")
            break

    # --- graceful finish: return the best partial result + the reason ------
    partial = _graceful_partial(goal, steps, breaker)
    return partial, meter, breaker


def _graceful_partial(goal, steps, breaker):
    if breaker.state != "tripped":
        return f"RUNAWAY (no guards): {len(steps)} steps and still not done."
    seen = len(steps)
    return (f"Stopped gracefully after {seen} steps ({breaker.reason}). "
            f"Partial result: could not complete '{goal}'; returning what was gathered "
            "instead of looping or crashing.")


print("run_agent, _graceful_partial ready.")

## Step 6 - `generate()`: the real LLM controller path (reference shape only)

Same device as Parts 1-7. In production the controller is an LLM: you hand it the goal and the running transcript, and it emits the next step. `generate()` is the single swappable call that lights up the real path; the offline demo never calls it, because the deterministic controllers above are the source of truth for everything in the output. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM for the next step. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM would drive the loop via generate(). "
          "Falling through to the deterministic controllers so output is reproducible.")
else:
    print("[controller] no OPENAI_API_KEY; using deterministic controllers (offline default)")

## Demo - the runaway, the loop-detector trip, then the budget trip

Everything below runs fully offline. The three runs share one set of ceilings and a loop threshold of 3, and use the same `run_agent` loop:

1. **No guards (runaway).** The stuck controller re-issues the identical search; with `guards=False` only the demo `hard_cap` stops it, after eight wasted steps. This is what an unguarded loop looks like.
2. **Guards on, stuck.** The same stuck controller, but now the loop detector trips the breaker at the threshold (3 identical actions), well under budget. The loop caught it before the budget did.
3. **Guards on, wandering.** The wandering controller never repeats, so the loop detector sees nothing; the multi-dimensional budget stops it instead, tripping the breaker before step 5 on the step ceiling.

In [ ]:
bar = "=" * 72
print(bar)
print("STOPPING THE RUNAWAY  -  budgets, loop detection, and the circuit breaker")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM would drive the loop via generate(). "
          "Falling through to the deterministic controllers so output is reproducible.")
else:
    print("[controller] no OPENAI_API_KEY; using deterministic controllers (offline default)")

CEILINGS = (4, 600, 0.02, 12.0)        # max_steps, max_tokens, max_usd, max_wall
print(f"\nBudget ceilings: {CEILINGS[0]} steps | {CEILINGS[1]} tokens | "
      f"${CEILINGS[2]:.2f} | {CEILINGS[3]:.0f}s (simulated). Loop threshold: 3 identical actions.")

### Run 1 - No guards: a stuck agent re-issues the same search and never stops

With `guards=False`, `run_agent` falls back to the plain `hard_cap`. The stuck controller asks for the identical discount-code search every step; the meter climbs while nothing is learned. The demo cap cuts it at 8 steps, but the point is that without guards the agent would not have stopped on its own. The final meter (8 steps, 640 tokens, $0.0064, ~9.6s) is the bill an unguarded loop runs up.

In [ ]:
print("\n" + "-" * 72)
print("1) NO GUARDS: a stuck agent re-issues the same search and never stops.")
print("-" * 72)
ans, meter, br = run_agent("Find a discount code for ORD-9999", stuck_controller,
                           CEILINGS, guards=False, hard_cap=8)
print(f"    final meter: {meter.snapshot()}")
print(f"    -> {ans}")

### Run 2 - Guards on: the loop detector catches the identical action at the threshold

Same stuck controller, but now `guards=True`. The loop detector counts the identical `(search_policy, query)` key and trips the breaker the moment it hits 3. The run stops at step 3 (240 tokens, $0.0024, ~3.6s), far under the budget; the loop caught it first. Instead of a crash or an endless spin, the agent returns a graceful partial result that names the reason it stopped.

In [ ]:
print("\n" + "-" * 72)
print("2) GUARDS ON: the loop detector catches the identical action at the threshold.")
print("-" * 72)
ans, meter, br = run_agent("Find a discount code for ORD-9999", stuck_controller,
                           CEILINGS, guards=True, loop_threshold=3)
print(f"    breaker: {br.state} ({br.reason})")
print(f"    final meter: {meter.snapshot()}  (well under budget; the loop caught it first)")
print(f"    -> {ans}")

### Run 3 - Guards on: a wandering agent never repeats, so the BUDGET stops it

The wandering controller issues a fresh "policy clause N" search every step, so no `(action, args)` key ever repeats and the loop detector stays silent. This is the failure a loop detector alone would miss. The multi-dimensional budget catches it: the pre-step guard sees the step ceiling (4) crossed and trips the breaker before step 5. The agent again returns a graceful partial result rather than running forever.

In [ ]:
print("\n" + "-" * 72)
print("3) GUARDS ON: a wandering agent never repeats, so the BUDGET stops it.")
print("-" * 72)
ans, meter, br = run_agent("Audit every policy clause", wandering_controller,
                           CEILINGS, guards=True, loop_threshold=3)
print(f"    breaker: {br.state} ({br.reason})")
print(f"    final meter: {meter.snapshot()}")
print(f"    -> {ans}")

print("\n" + bar)
print("Done. A single max_steps cap cannot tell a stuck run from an expensive one.")
print("  - a MULTI-DIMENSIONAL budget (steps, tokens, cost, time) stops the first to cross")
print("  - a LOOP DETECTOR catches the identical-action signature (formalizes P19's note)")
print("  - a CIRCUIT BREAKER trips to a GRACEFUL partial result, never a crash or a spin")
print("Owned here, reused later: the BudgetMeter in Part 13, the breaker in Part 14.")
print(bar)

## Wrap-up

Part 7 let the agent survive the window and the store. It still had a third way to fail: it can loop, oscillate, or burn unbounded steps and tokens chasing a goal it will never reach. A single `max_steps` cap, the one guard carried from RAG Part 19, cannot tell a stuck run from an expensive one, and it ignores the dimensions that actually cost money.

- **The multi-dimensional budget** (`BudgetMeter`) tracks steps, tokens, cost, and simulated time, each with a ceiling checked before every step; the first dimension to cross stops the run. The wandering agent, which no loop detector would catch, was stopped here on the step ceiling.
- **The loop detector** formalizes RAG Part 19's informal "saw the same search twice" note into a real `(action, args)` repeat counter. The stuck agent was caught at the threshold, far under budget.
- **The circuit breaker** turns either signal into a graceful finish: closed -> tripped -> a partial result plus the reason, never a crash and never a spin.

Remember the honesty caveats: wall-clock is simulated, the cost numbers are illustrative, and the runaway framing is generalized with no fabricated dollar figure. And remember the reuse: the `BudgetMeter` returns in **Part 13** to cap a code-execution sandbox, and the circuit breaker in **Part 14** to stop a runaway supervisor. We built them cleanly here for exactly that reason.

**Part 9 - The durable agent** is next. So far every run lives and dies in one process: crash midway and the transcript, the budget, and the partial result are gone. Part 9 makes the agent durable, so a long run can be checkpointed, interrupted, and resumed without starting over.